In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import lightgbm as lgb


In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(train.shape, test.shape)
print("Train days:", sorted(train['day'].unique()))
print("Test  days:", sorted(test['day'].unique()))
print("Test hours:", sorted(test['timestamp'].str.split(':').str[0].astype(int).unique()))


In [ ]:
def preprocess(df):
    df = df.copy()
    df['hour']         = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']       = df['timestamp'].str.split(':').str[1].astype(int)
    df['time_minutes'] = df['hour'] * 60 + df['minute']
    
    # Cyclical Time
    T = 24 * 60
    df['sin_time']     = np.sin(2 * np.pi * df['time_minutes'] / T)
    df['cos_time']     = np.cos(2 * np.pi * df['time_minutes'] / T)
    
    # Categorical Encodings
    df['RoadType_enc']      = df['RoadType'].map({'Residential':0,'Street':1,'Highway':2}).fillna(-1)
    df['Weather_enc']       = df['Weather'].map({'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3}).fillna(-1)
    df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks_enc']     = (df['Landmarks'] == 'Yes').astype(int)
    
    # Spatial Hierarchies
    df['geo_prefix3']  = df['geohash'].str[:3]
    df['geo_prefix4']  = df['geohash'].str[:4]
    
    # Advanced Temporal Features
    if 'day' in df.columns:
        df['day_of_week'] = df['day'] % 7
    else:
        df['day_of_week'] = 0
        
    df['is_rush_hour'] = df['hour'].isin([8, 9, 17, 18, 19]).astype(int)
    return df

train = preprocess(train)
test  = preprocess(test)

# Safe Imputation: Fit on Train only, apply to both
train_temp_median = train['Temperature'].median()
train['Temperature'] = train['Temperature'].fillna(train_temp_median)
test['Temperature']  = test['Temperature'].fillna(train_temp_median)

global_mean = float(train['demand'].mean())


In [ ]:
# 1. Combine Train and Test for continuous historical lags across midnight boundaries
train['is_test'] = 0
test['is_test']  = 1
df_all = pd.concat([train, test], ignore_index=True)

# Sort globally by space and exact time sequence
df_all = df_all.sort_values(['geohash', 'day', 'time_minutes']).reset_index(drop=True)

# Group strictly by geohash (ignoring day) to maintain continuous lag flow
grp = df_all.groupby('geohash')['demand']
df_all['demand_lag1']  = grp.shift(1)
df_all['demand_lag2']  = grp.shift(2)
df_all['demand_lag4']  = grp.shift(4)
df_all['demand_roll3'] = grp.shift(1).rolling(3, min_periods=1).mean().reset_index(0, drop=True)

# Fallback NaNs for very first observations
geo_mean_s = train.groupby('geohash')['demand'].mean()
for col in ['demand_lag1','demand_lag2','demand_lag4','demand_roll3']:
    df_all[col] = df_all[col].fillna(df_all['geohash'].map(geo_mean_s)).fillna(global_mean)


# 2. Out-Of-Fold (OOF) Target Encoding to eliminate data leakage
df_all['geo_hour_mean'] = np.nan
df_all['geo_mean']      = np.nan

train_idx = df_all[df_all['is_test'] == 0].index
test_idx  = df_all[df_all['is_test'] == 1].index

# A. For test set: Use the entire train set's means
gh_mean_full = df_all.loc[train_idx].groupby(['geohash', 'hour'])['demand'].mean()
g_mean_full  = df_all.loc[train_idx].groupby('geohash')['demand'].mean()

df_all.loc[test_idx, 'geo_hour_mean'] = df_all.loc[test_idx].set_index(['geohash', 'hour']).index.map(gh_mean_full).values
df_all.loc[test_idx, 'geo_mean']      = df_all.loc[test_idx, 'geohash'].map(g_mean_full).values

# B. For train set: Use 5-Fold OOF
kf = KFold(n_splits=5, shuffle=True, random_state=42)
train_df = df_all.loc[train_idx].copy()

for trn_fold_idx, val_fold_idx in kf.split(train_df):
    trn_fold = train_df.iloc[trn_fold_idx]
    val_fold = train_df.iloc[val_fold_idx]
    
    gh_fold_mean = trn_fold.groupby(['geohash', 'hour'])['demand'].mean()
    g_fold_mean  = trn_fold.groupby('geohash')['demand'].mean()
    
    # Map folds safely back to main dataframe index mapping
    original_val_idx = train_idx[val_fold_idx]
    df_all.loc[original_val_idx, 'geo_hour_mean'] = val_fold.set_index(['geohash', 'hour']).index.map(gh_fold_mean).values
    df_all.loc[original_val_idx, 'geo_mean']      = val_fold['geohash'].map(g_fold_mean).values

# Impute remaining unmatched combinations
df_all['geo_mean']      = df_all['geo_mean'].fillna(global_mean)
df_all['geo_hour_mean'] = df_all['geo_hour_mean'].fillna(df_all['geo_mean'])

# 3. Split the cleaned data back apart
train = df_all[df_all['is_test'] == 0].drop(columns=['is_test']).copy()
test  = df_all[df_all['is_test'] == 1].drop(columns=['is_test']).copy()


In [ ]:
# Use entire history (days <= 48) for robust training, instead of just day 48
train_48 = train[train['day'] <= 48]
val_49   = train[train['day'] == 49]

FEATURES = [
    'day','hour','minute','time_minutes','sin_time','cos_time','day_of_week','is_rush_hour',
    'RoadType_enc','NumberofLanes','LargeVehicles_enc','Landmarks_enc',
    'Temperature','Weather_enc',
    'geo_mean', 'geo_hour_mean',
    'demand_lag1','demand_lag2','demand_lag4','demand_roll3'
]

X_48   = train_48[FEATURES].values;   y_48  = train_48['demand'].values
X_49   = val_49[FEATURES].values;     y_49  = val_49['demand'].values
X_all  = train[FEATURES].values;      y_all = train['demand'].values
X_test = test[FEATURES].values

print(f"X_train:{X_48.shape}  X_val:{X_49.shape}  X_all:{X_all.shape}  X_test:{X_test.shape}")
print(f"NaN check — train:{np.isnan(X_all).sum()}  test:{np.isnan(X_test).sum()}")


In [ ]:
params = dict(
    n_estimators=2000, learning_rate=0.05, max_depth=7,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=15, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbose=-1
)

m_val = lgb.LGBMRegressor(**params)
m_val.fit(X_48, y_48,
          eval_set=[(X_49, y_49)],
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(50)])

val_score = max(0, 100 * r2_score(y_49, m_val.predict(X_49)))
best_iter  = m_val.best_iteration_
print(f"\nHonest validation score (Days <= 48 → Day 49): {val_score:.2f} / 100")
print(f"Best iteration: {best_iter}")


In [ ]:
imp = pd.Series(m_val.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind='barh', figsize=(8,7), title='Feature importance')
plt.tight_layout()
plt.show()


In [ ]:
params_final = params.copy()
params_final['n_estimators'] = best_iter + 50

m_final = lgb.LGBMRegressor(**params_final)
m_final.fit(X_all, y_all)

preds = np.clip(m_final.predict(X_test), 0, 1)

submission = pd.DataFrame({
    'Index' : test['Index'].values,
    'demand': preds
}).sort_values('Index').reset_index(drop=True)

submission.to_csv('submission_fixed.csv', index=False)
print(f"Saved — {len(submission)} rows")
print(f"Validation score: {val_score:.2f} / 100")
submission.head()
